# House Prices - Advanced Regression Techniques
**Kaggle Public Score: 0.12033 | 324th place**

## Pipeline
1. Data Loading & Missing Value Imputation
2. ID Separation
3. Outlier Removal
4. Feature Engineering (numerical + categorical)
5. Target Encoding (high-cardinality categoricals)
6. Skewness Transform
7. Preprocessing (StandardScaler + OneHotEncoder)
8. Feature Selection (intersection of model importances > 0 → 79 features)
9. Stacking Ensemble (LGB + XGB + GBM + CatBoost + ElasticNet)
10. Final Prediction & Submission

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
from scipy.stats import skew
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import GradientBoostingRegressor, StackingRegressor
from sklearn.linear_model import ElasticNet, RidgeCV
from sklearn.model_selection import cross_validate
import lightgbm as lgb
import xgboost as xgb
from catboost import CatBoostRegressor
import category_encoders as ce

## 1. Data Loading & Missing Value Imputation
결측치 처리 파이프라인

### 결측치 처리 전략
- **1단계 (논리적 정제)**: 데이터 문서 기반으로 컬럼별 결측 의미 파악 후 처리
- **2단계 (통계적 처리)**: train 기준으로 남은 결측치 일괄 처리 (No Leakage)
- **3단계 (구조 정합성)**: train/test 컬럼 동기화 및 최종 결측치 소거
- **4단계**: 이산형 → 범주형 변환

In [ ]:
import os

train_path = os.path.join(os.getcwd(), '../data/raw/train.csv')
test_path = os.path.join(os.getcwd(), '../data/raw/test.csv')
train = pd.read_csv(train_path)
test = pd.read_csv(test_path)

### 1단계: 논리적 정제 (Logical Cleaning)
각 컬럼의 결측치가 가진 **실제 의미**를 데이터 문서 기반으로 판단하여 처리

In [ ]:
def fix_lotfrontage_with_regression(train_df, test_df):
    """
    LotFrontage: 1stFlrSF와의 선형 관계를 활용해 결측치 예측
    - 결측 여부를 LotFrontage_flag로 기록
    - train 모델로 test 결측치도 예측 (No Leakage)
    """
    from sklearn.linear_model import LinearRegression

    train_df['LotFrontage_flag'] = train_df['LotFrontage'].isna().astype(int)
    test_df['LotFrontage_flag']  = test_df['LotFrontage'].isna().astype(int)

    has_val      = train_df[train_df['LotFrontage'].notna()].copy()
    train_nan    = train_df[train_df['LotFrontage'].isna()].copy()
    train_refined = has_val[has_val['LotFrontage'] <= 200]

    X_train = np.log1p(train_refined[['1stFlrSF']])
    Y_train = np.log1p(train_refined['LotFrontage'])
    model = LinearRegression()
    model.fit(X_train, Y_train)

    if not train_nan.empty:
        X_nan = np.log1p(train_nan[['1stFlrSF']])
        train_df.loc[train_df['LotFrontage'].isna(), 'LotFrontage'] = np.expm1(model.predict(X_nan))

    test_nan = test_df[test_df['LotFrontage'].isna()].copy()
    if not test_nan.empty:
        X_test_nan = np.log1p(test_nan[['1stFlrSF']])
        test_df.loc[test_df['LotFrontage'].isna(), 'LotFrontage'] = np.expm1(model.predict(X_test_nan))

    return train_df, test_df

In [ ]:
def fix_alley_all(train_df, test_df):
    """Alley: NA = 골목길 없음 (문서 명시)"""
    def apply(df):
        return df.copy().fillna({'Alley': 'NA'})
    return apply(train_df), apply(test_df)


def fix_masvnr_all(train_df, test_df):
    """
    MasVnr: 타입/면적 결측 조합별 논리 처리
    - 타입 NaN & 면적 > 0 → Unknown (추측, 플래그 ON)
    - 타입 NaN & 면적 NaN or 0 → None/0 (없음 확정)
    - 타입 있는데 면적 0 → train 평균으로 보정 (플래그 ON)
    """
    all_types = train_df['MasVnrType'].dropna().unique()
    avg_map   = {t: train_df[train_df['MasVnrType'] == t]['MasVnrArea'].replace(0, np.nan).mean()
                 for t in all_types}
    total_avg = train_df['MasVnrArea'].replace(0, np.nan).mean()

    def apply(df):
        df = df.copy()
        for col in ['MasVnrType_flag', 'MasVnrArea_flag']:
            if col not in df.columns:
                df[col] = 0
        mask1 = df.MasVnrType.isna() & (df.MasVnrArea > 0)
        df.loc[mask1, ['MasVnrType_flag', 'MasVnrType']] = [1, 'Unknown']
        mask2 = df.MasVnrType.isna() & df.MasVnrArea.isna()
        df.loc[mask2, ['MasVnrType', 'MasVnrArea']] = ['None', 0]
        mask3 = df.MasVnrType.isna() & (df.MasVnrArea == 0)
        df.loc[mask3, 'MasVnrType'] = 'None'
        for t, avg in avg_map.items():
            mask = (df['MasVnrType'] == t) & (df['MasVnrArea'] == 0)
            df.loc[mask, 'MasVnrArea_flag'] = 1
            df.loc[mask, 'MasVnrArea'] = avg if not np.isnan(avg) else 0
        return df

    return apply(train_df), apply(test_df)

In [ ]:
def fix_bsmt_all(train_df, test_df):
    """
    Basement: 지하실 없는 집(TotalBsmtSF=0)은 모든 범주형을 NA로
    예외: BsmtExposure/BsmtFinType2 결측이면서 지하실 있는 경우 → Unknown
    """
    def apply(df):
        df = df.copy()
        for col in ['BsmtExposure_flag', 'BsmtFinType2_flag']:
            if col not in df.columns:
                df[col] = 0
        mask1 = df.BsmtExposure.isna() & df.BsmtQual.notna()
        df.loc[mask1, ['BsmtExposure_flag', 'BsmtExposure']] = [1, 'Unknown']
        mask2 = (df['BsmtFinSF2'] != 0) & df['BsmtFinType2'].isna()
        df.loc[mask2, ['BsmtFinType2_flag', 'BsmtFinType2']] = [1, 'Unknown']
        mask3 = df['TotalBsmtSF'] == 0
        df.loc[mask3, ['BsmtQual','BsmtCond','BsmtExposure','BsmtFinType1','BsmtFinType2']] = 'NA'
        return df

    return apply(train_df), apply(test_df)


def fix_elect_all(train_df, test_df):
    """
    Electrical: 1970년 이후 건축 주택은 표준 회로(SBrkr)로 추정
    """
    def apply(df):
        df = df.copy()
        if 'Electrical_flag' not in df.columns:
            df['Electrical_flag'] = 0
        mask = df['Electrical'].isna()
        df.loc[mask, 'Electrical_flag'] = 1
        df.loc[mask & (df['YearBuilt'] >= 1970), 'Electrical'] = 'SBrkr'
        return df

    return apply(train_df), apply(test_df)


def fix_fireplacequ_all(train_df, test_df):
    """FireplaceQu: 벽난로 없는 집(Fireplaces=0)은 NA"""
    def apply(df):
        df = df.copy()
        mask = (df['Fireplaces'] == 0) & df['FireplaceQu'].isna()
        df.loc[mask, 'FireplaceQu'] = 'NA'
        return df

    return apply(train_df), apply(test_df)


def fix_garage_all(train_df, test_df):
    """
    Garage: 주차장 없는 집은 범주형 NA, GarageYrBlt는 YearBuilt로 대체
    """
    def apply(df):
        df = df.copy()
        if 'GarageYrBlt_flag' not in df.columns:
            df['GarageYrBlt_flag'] = 0
        mask = ((df['GarageCars'] == 0) & (df['GarageArea'] == 0)
                & df['GarageType'].isna() & df['GarageYrBlt'].isna()
                & df['GarageFinish'].isna() & df['GarageQual'].isna()
                & df['GarageCond'].isna())
        df.loc[mask, ['GarageType','GarageFinish','GarageQual','GarageCond']] = 'NA'
        df.loc[mask, 'GarageYrBlt_flag'] = 1
        df.loc[mask, 'GarageYrBlt'] = df['YearBuilt']
        return df

    return apply(train_df), apply(test_df)


def fix_poolqc_all(train_df, test_df):
    """PoolQC: 수영장 없는 집(PoolArea=0)은 NA"""
    def apply(df):
        df = df.copy()
        df.loc[(df['PoolArea'] == 0) & df['PoolQC'].isna(), 'PoolQC'] = 'NA'
        return df

    return apply(train_df), apply(test_df)


def fix_fence_all(train_df, test_df):
    """Fence: 결측 = 울타리 없음"""
    def apply(df):
        return df.copy().fillna({'Fence': 'NA'})
    return apply(train_df), apply(test_df)


def fix_misc_all(train_df, test_df):
    """
    MiscFeature: 이상 거래(Abnorml/Family)에서 MiscVal=0이면 Unknown
    정상 거래에서 MiscVal=0이면 오류로 간주하여 NA
    """
    def apply(df):
        df = df.copy()
        if 'MiscFeature_flag' not in df.columns:
            df['MiscFeature_flag'] = 0
        mask_base     = df['MiscFeature'].notna() & (df['MiscVal'] == 0)
        mask_abnormal = mask_base & df['SaleCondition'].isin(['Abnorml','Family'])
        mask_normal   = mask_base & ~df['SaleCondition'].isin(['Abnorml','Family'])
        df.loc[mask_abnormal, ['MiscFeature_flag', 'MiscFeature']] = [1, 'Unknown']
        df.loc[mask_normal, 'MiscFeature'] = 'NA'
        df = df.fillna({'MiscFeature': 'NA'})
        return df

    return apply(train_df), apply(test_df)

In [ ]:
# 1단계 순차 적용
train_df, test_df = fix_lotfrontage_with_regression(train, test)
train_df, test_df = fix_alley_all(train_df, test_df)
train_df, test_df = fix_masvnr_all(train_df, test_df)
train_df, test_df = fix_bsmt_all(train_df, test_df)
train_df, test_df = fix_elect_all(train_df, test_df)
train_df, test_df = fix_fireplacequ_all(train_df, test_df)
train_df, test_df = fix_garage_all(train_df, test_df)
train_df, test_df = fix_poolqc_all(train_df, test_df)
train_df, test_df = fix_fence_all(train_df, test_df)
train_df, test_df = fix_misc_all(train_df, test_df)
print('1단계 완료')

### 2단계: 통계적 일괄 처리 (Statistical Imputation)
train 기준값(수치형=중앙값, 범주형=최빈값)으로 test 결측치 처리 — No Leakage

In [ ]:
def apply_statistical_impute(train_df, test_df):
    """
    train 기준으로 각 컬럼의 대표값을 추출하여 test 결측치에만 적용
    수치형: 중앙값 / 범주형: 최빈값
    """
    train_df = train_df.copy()
    test_df  = test_df.copy()
    impute_values = {}
    for col in train_df.columns:
        if pd.api.types.is_numeric_dtype(train_df[col]):
            impute_values[col] = train_df[col].median()
        else:
            mode_series = train_df[col].mode()
            impute_values[col] = mode_series[0] if not mode_series.empty else 'NA'
    test_df = test_df.fillna(value=impute_values)
    for col in train_df.columns:
        if col in test_df.columns:
            test_df[col] = test_df[col].astype(train_df[col].dtype)
    return train_df, test_df

train_df, test_df = apply_statistical_impute(train_df, test_df)

# 결측치 확인
remaining = test_df.isnull().sum()
print(remaining[remaining > 0] if remaining.sum() > 0 else '결측치 없음 ✅')

### 3단계: 구조적 정합성 (Structural Alignment)
train/test 컬럼 구성 및 순서 동기화

In [ ]:
def finalize_data_structure(train_df, test_df):
    """
    1. test에만 있는 불필요한 컬럼 삭제
    2. train에만 있는 컬럼을 test에 추가 (수치형=0, 범주형='NA')
    3. 컬럼 순서 일치
    """
    train_df = train_df.copy()
    test_df  = test_df.copy()
    extra   = set(test_df.columns) - set(train_df.columns)
    test_df = test_df.drop(columns=list(extra))
    missing = set(train_df.columns) - set(test_df.columns)
    for col in missing:
        test_df[col] = 0 if train_df[col].dtype != 'O' else 'NA'
    test_df = test_df[train_df.columns]
    return train_df, test_df

train_df, test_df = finalize_data_structure(train_df, test_df)

print(f'컬럼 일치: {train_df.columns.equals(test_df.columns)}')
print(f'Train 결측치: {train_df.isnull().sum().sum()}')
print(f'Test  결측치: {test_df.isnull().sum().sum()}')

### 4단계: 이산형 → 범주형 변환
숫자지만 크기가 의미 없는 컬럼(MSSubClass, MoSold) 문자열로 변환

In [ ]:
def fix_discrete_all(train_df, test_df):
    cols_to_str = ['MSSubClass', 'MoSold']
    def apply(df):
        df = df.copy()
        for col in cols_to_str:
            df[col] = df[col].astype(str)
        return df
    return apply(train_df), apply(test_df)

train_df, test_df = fix_discrete_all(train_df, test_df)

# 최종 저장
train_df.to_csv('/kaggle/working/train_df.csv', index=False)
test_df.to_csv('/kaggle/working/test_df.csv', index=False)
print('✅ train_df.csv / test_df.csv 저장 완료')

## 2. ID Separation

In [ ]:
train_df = train_df.drop(columns='Id')
test_ids = test_df['Id']
test_df  = test_df.drop(columns='Id')

## 3. Outlier Removal
상관계수/상관비 분석 결과 기반으로 극단적 이상치 제거

In [3]:
train_df = train_df[train_df['GrLivArea']   <= 4500]
train_df = train_df[train_df['LotArea']     <= 200000]
train_df = train_df[train_df['LotFrontage'] <= 300]

## 4. Feature Engineering
### 4.1 Numerical Features
상관계수 분석으로 최적 가중치를 도출한 파생 피처

In [4]:
def nums(df):
    # 거주 면적: 1층/2층 가중 합산 (상관계수 분석 기반)
    df['new_GrLivArea'] = df['1stFlrSF'] * 1.2 + df['2ndFlrSF'] * 0.7
    # 외부 공간: 포치/덱 가중 합산
    df['Porch_Deck'] = (df['OpenPorchSF'] * 1.05
                      + df['3SsnPorch']   * 0.37
                      + df['ScreenPorch'] * 0.37
                      + df['WoodDeckSF']  * 0.58)
    # 주택 연령
    df['HouseAge'] = df['YrSold'] - df['YearBuilt']
    return df

train_df = nums(train_df)
test_df  = nums(test_df)

train_df = train_df.drop(columns=['YrSold', 'YearBuilt'])
test_df  = test_df.drop(columns=['YrSold', 'YearBuilt'])

### 4.2 Categorical Features
순서형 범주: 등급 → 숫자 변환  
이진 플래그: 정상/부분 거래 여부

In [5]:
# 정상/부분 거래 여부 플래그
train_df['SaleCondition_flag'] = train_df['SaleCondition'].isin(['Normal', 'Partial']).astype(int)
test_df['SaleCondition_flag']  = test_df['SaleCondition'].isin(['Normal', 'Partial']).astype(int)

# 품질 등급 → 숫자 (Ex=5, Gd=4, TA=3, Fa=2, Po=1, NA=0)
def replace_label_case1(df, col):
    grading_map = {'Ex': 5, 'Gd': 4, 'TA': 3, 'Fa': 2, 'Po': 1, 'NA': 0}
    df[col] = df[col].map(grading_map).fillna(0)
    return df

# FireplaceQu: Po=0 (없음과 구분)
def replace_label_case2(df, col):
    grading_map = {'Ex': 5, 'Gd': 4, 'TA': 3, 'Fa': 2, 'Po': 0, 'NA': 1}
    df[col] = df[col].map(grading_map).fillna(0)
    return df

columns_case1 = ['GarageQual', 'HeatingQC', 'PoolQC', 'ExterQual',
                 'BsmtQual', 'KitchenQual', 'GarageCond', 'ExterCond']
columns_case2 = ['BsmtCond', 'FireplaceQu']

for col in columns_case1:
    replace_label_case1(train_df, col)
    replace_label_case1(test_df, col)
for col in columns_case2:
    replace_label_case2(train_df, col)
    replace_label_case2(test_df, col)

## 5. Target & Target Encoding
- Target: log1p(SalePrice) → 왜도 정규화, RMSLE 메트릭과 일치
- Target Encoding: 고카디널리티 범주형 → 타깃 평균값으로 변환
- **No Leakage**: train 데이터로만 fit, test는 transform만 적용

In [6]:
X_train = train_df.drop(columns='SalePrice')
y_train = train_df['SalePrice']
y_log   = np.log1p(y_train)

# 카디널리티 높은 범주형 컬럼 Target Encoding
# Neighborhood(25), Exterior1st(15), Exterior2nd(16), MSSubClass(15)
target_enc_cols = ['Neighborhood', 'Exterior1st', 'Exterior2nd', 'MSSubClass']

te = ce.TargetEncoder(cols=target_enc_cols, smoothing=10)
te.fit(X_train[target_enc_cols], y_log)

X_train[target_enc_cols] = te.transform(X_train[target_enc_cols])
test_df[target_enc_cols]  = te.transform(test_df[target_enc_cols])

## 6. Skewness Transform
왜도 0.75 이상인 수치형 피처에 log1p 적용

In [7]:
num_cols    = X_train.select_dtypes(include='number').columns
skewed_cols = X_train[num_cols].apply(skew).abs()
skewed_cols = skewed_cols[skewed_cols >= 0.75].index

X_train[skewed_cols] = np.log1p(X_train[skewed_cols])
test_df[skewed_cols]  = np.log1p(test_df[skewed_cols])

## 7. Preprocessing Pipeline
- 수치형: StandardScaler
- 범주형: OneHotEncoder

In [8]:
def num_cat_features(df):
    num_features = df.select_dtypes(include='number').columns
    cat_features = df.select_dtypes(include=['object', 'string']).columns
    return num_features, cat_features

num_features, cat_features = num_cat_features(X_train)

preprocessor = ColumnTransformer(transformers=[
    ('num', StandardScaler(), num_features),
    ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), cat_features)
])

X_processed       = preprocessor.fit_transform(X_train)
test_df_processed  = preprocessor.transform(test_df)
feature_names     = preprocessor.get_feature_names_out()

## 8. Feature Selection
4개 모델의 Feature Importance 교집합 → 모든 모델이 공통으로 중요하게 본 피처만 사용

In [9]:
def get_importance_features(model, feature_names):
    importance_df = pd.DataFrame({
        'Feature': feature_names,
        'Importance': model.feature_importances_
    })
    return set(importance_df[importance_df['Importance'] > 0]['Feature'])

lgb_model = lgb.LGBMRegressor(random_state=42, verbose=-1)
xgb_model = xgb.XGBRegressor(random_state=42)
gbm_model = GradientBoostingRegressor(random_state=42)
cat_model = CatBoostRegressor(random_state=42, verbose=0)

for model in [lgb_model, xgb_model, gbm_model, cat_model]:
    model.fit(X_processed, y_log)

lgb_features      = get_importance_features(lgb_model, feature_names)
xgb_features      = get_importance_features(xgb_model, feature_names)
gbm_features      = get_importance_features(gbm_model, feature_names)
catboost_features = get_importance_features(cat_model, feature_names)

inter_features = list(lgb_features & xgb_features & gbm_features & catboost_features)
print(f'선택된 피처 수: {len(inter_features)}개')

X_processed_df    = pd.DataFrame(X_processed,        columns=feature_names)
test_processed_df  = pd.DataFrame(test_df_processed,  columns=feature_names)

X_selected        = X_processed_df[inter_features]
test_df_selected  = test_processed_df[inter_features]

선택된 피처 수: 79개


## 9. Stacking Ensemble
- Base models: LGB, XGB, GBM, CatBoost (트리) + ElasticNet (선형)
- Meta model: RidgeCV
- cv=5: OOF 예측으로 Data Leakage 방지

In [10]:
lgb_best_params     = {'n_estimators': 800, 'learning_rate': 0.03624316233201206, 'max_depth': 3, 'num_leaves': 75, 'min_child_samples': 21, 'subsample': 0.8111635342652639, 'colsample_bytree': 0.6304831434719692, 'reg_alpha': 0.0010918991315994693, 'reg_lambda': 0.0051497538863298255}
xgb_best_params     = {'n_estimators': 976, 'learning_rate': 0.028194901372253063, 'max_depth': 3, 'min_child_weight': 5, 'subsample': 0.7288866619554035, 'colsample_bytree': 0.6316772621483024, 'reg_alpha': 0.13787183973684694, 'reg_lambda': 0.47062910249812745}
gbm_best_params     = {'n_estimators': 941, 'learning_rate': 0.02743211204415027, 'max_depth': 3, 'subsample': 0.6872297621992344, 'max_features': 0.4602375943728553, 'min_samples_leaf': 5}
cat_best_params     = {'iterations': 994, 'depth': 5, 'learning_rate': 0.043692556025664915, 'subsample': 0.6068969053674399, 'l2_leaf_reg': 1.3271915791457247}
elastic_best_params = {'alpha': 0.0006408925745171936, 'l1_ratio': 0.884798640263611}

best_lgb     = lgb.LGBMRegressor(**lgb_best_params, random_state=42, verbose=-1)
best_xgb     = xgb.XGBRegressor(**xgb_best_params, random_state=42)
best_gbm     = GradientBoostingRegressor(**gbm_best_params, random_state=42)
best_cat     = CatBoostRegressor(**cat_best_params, random_seed=42, verbose=0)
best_elastic = ElasticNet(**elastic_best_params, max_iter=10000)

stack_model = StackingRegressor(
    estimators=[
        ('lgb',     best_lgb),
        ('xgb',     best_xgb),
        ('gbm',     best_gbm),
        ('cat',     best_cat),
        ('elastic', best_elastic),
    ],
    final_estimator=RidgeCV(),
    cv=5,
    n_jobs=-1
)

cv_results = cross_validate(
    stack_model, X_selected, y_log,
    cv=5,
    scoring={'mse': 'neg_mean_squared_error'}
)
rmsle = np.sqrt(-cv_results['test_mse']).mean()
print(f'Stacking CV RMSLE: {rmsle:.4f}')

Stacking CV RMSLE: 0.1107


## 10. Final Prediction & Submission

In [ ]:
stack_model.fit(X_selected, y_log)

pred_stack = np.expm1(stack_model.predict(test_df_selected))

submission = pd.DataFrame({'Id': test_ids, 'SalePrice': pred_stack})
submission.to_csv('submission_house_prices_final.csv', index=False)

print(f'Prediction summary:')
print(f"  Min   : ${submission['SalePrice'].min():,.0f}")
print(f"  Max   : ${submission['SalePrice'].max():,.0f}")
print(f"  Mean  : ${submission['SalePrice'].mean():,.0f}")
print('✅ submission_house_prices_final.csv 저장 완료')

Prediction summary:
  Min   : $46,468
  Max   : $665,989
  Mean  : $178,679
✅ submission_house_prices_final.csv 저장 완료
